# O+O at RHIC: what is selection bias, what is quenching

The main analysis of the O+O physics run (phase 3 of the tuning page's "Physics run plan").
The numbers come from `oo_analysis_rhic.py` (`results/oo_analysis_rhic.json`); this notebook only
plots and tabulates them. **Model results, not vetted physics results.**

**The samples** (`oo_bias.py`, tune XMLs of js-contrib `oo_rhic_run` `073085b` and later):

| sample | what | role |
|---|---|---|
| `oo_vac_w*` | O+O, the jet in vacuum (`--vacuum-jet`: Matter in vacuum, no LBT; the medium is made) | selection only |
| `oo_med_w*` | O+O, the jet in the medium (Matter + LBT in MUSIC), the same seeds | selection × quenching |
| `pp_ref_w*` | p+p in the full chain (`config/pp200_isr_rhictune.xml`), jet in vacuum | the R_AA reference |
| `pp_med_w*` | p+p, the jet in p+p's own medium (3c) | consistent reference, Matter's QS check |
| `phys_mb` | O+O minimum bias, bulk hadrons only | class edges, ⟨N_coll⟩ per class |

Common settings: soft MPI (pT0Ref 1.48), C_v 0.25, the RHIC bulk tune; p̂T windows 5–80 GeV,
stitched by their cross sections; MUSIC + iSS in every job.

**How the conclusion is drawn.** Per centrality class c of an estimator (STAR's EPD, sPHENIX's MBD,
mid-rapidity, or the impact parameter as truth):
- **Selection bias** = R_AA^c of `oo_vac` (the jet saw no medium; ≠ 1 only because jet events are
  sorted differently from minimum bias).
- **Quenching in the classes** Q^c = R_AA^c(`oo_med`) / R_AA^c(`oo_vac`) (the same classes; the
  normalisations cancel).
- **Total** = R_AA^c(`oo_med`) = selection × quenching, the quantity to compare with data.
- **Geometry check**: Q in impact-parameter classes (Q_b) against Q in the measured classes
  (Q_mult): whether the class sorting distorts the quenching itself.
- **I_CP** (STAR): per-trigger yields, the same split into vacuum (selection) and medium (total).

**The medium's holes.** iColored hadronizes the recoils and, for holes it cannot close, a colour
partner with the hole's three-momentum, but drops the holes: the hadrons hold recoil + partner where
the balance is recoil − hole.
- **Jets**: subtracted at parton level, the holes (`partons/`, pstat −1) and the partner copies
  (`hadronization_input/`) inside the cone: sPHENIX full jets with their full four-momentum, STAR's
  charged jets (inclusive and recoil) with the charged fraction 2/3 of their transverse momentum.
  The unsubtracted spectra are kept (`*_raw`).
- **Hadron-level observables (the charged-hadron spectra, R_AA and R_CP, and the h–h I_CP) are not
  hole-subtracted.** They need `prod_OO_isr/negative_hadrons.py` (the holes and partners hadronized
  as negative hadrons, counted with weight −1), not yet run on the phase-3 medium files. Holes are
  soft, so the effect should be small above ≈ 4 GeV, but it is not quantified; until then these
  quenched hadron results carry that caveat.

Statistics: medium and vacuum share seeds but not events (LBT shifts the random stream), so every
ratio of two samples has independent errors. Matter's vacuum cutoff differs between the arms (QS 0.9
with the jet in the medium, Q0 2 in vacuum); its size comes from `pp_med`'s events without a medium
(section 6).

In [ ]:
import json, os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

R = json.loads(Path(os.environ.get("OO_ANALYSIS_RHIC_JSON", "results/oo_analysis_rhic.json")).read_text())
E = {k: np.asarray(v) for k, v in R["edges"].items()}
print("inputs:", R["inputs"]["out"], "| minimum bias:", R["inputs"]["mb_root"], R["inputs"]["mb_events"], "events")
for f, n in R["inputs"]["events"].items():
    print(f"  {f:7s} {n:8d} events")
OBS = {"had": ("charged hadrons, |η| < 1 (holes not subtracted)", "$p_T$ [GeV]", E["had"]),
       "sph_jet": ("sPHENIX full jets R = 0.4, |η| < 0.7 (holes + partners subtracted)", "$p_{T,jet}$ [GeV]", E["sph_jet"]),
       "ch_jet_R0.2": ("STAR charged jets R = 0.2 (2/3 of holes + partners subtracted)", "$p_{T,jet}^{ch}$ [GeV]", E["ch_jet"]),
       "ch_jet_R0.5": ("STAR charged jets R = 0.5 (2/3 of holes + partners subtracted)", "$p_{T,jet}^{ch}$ [GeV]", E["ch_jet"])}
EST = {"epd": "STAR EPD", "mbd": "sPHENIX MBD", "mid10": "mid-rapidity |η|<1", "b": "impact parameter (truth)"}
COLS = plt.cm.viridis(np.linspace(0, 0.9, 6))

def xc(e):
    return 0.5 * (e[1:] + e[:-1]), 0.5 * (e[1:] - e[:-1])

def plot_raa(ax, block, fam, edges, title, which="raa"):
    x, xe = xc(edges)
    for c, (lo, hi) in enumerate(block["classes"]):
        y = np.asarray(block[fam][which][c], float); e = np.asarray(block[fam]["err"][c], float)
        ax.errorbar(x + (c - 2.5) * 0.08 * xe, y, e, xe, fmt="o", ms=3, color=COLS[c], label=f"{lo}-{hi}%")
    ax.axhline(1, color="k", lw=0.8); ax.set_title(title, fontsize=9); ax.set_ylim(0, 2)

## 1. Selection bias: R_AA of the vacuum jets per class

The jet never sees the medium, so any deviation from 1 is the selection: jet events are sorted into
the classes differently from minimum-bias events (the hard-momentum subtraction, the MPI, the jet's
own hadrons in the estimator). The impact-parameter row is the truth check (no selection by
construction; it should be 1 within the statistics, apart from what the hard-momentum subtraction
does to the inclusive spectrum).

In [ ]:
if R["raa"]:
    fig, ax = plt.subplots(len(EST), len(OBS), figsize=(16, 3.2 * len(EST)), sharey=True)
    for i, est in enumerate(EST):
        for j, (key, (lab, xl, edges)) in enumerate(OBS.items()):
            blk = R["raa"][est][key]
            if "oo_vac" in blk:
                plot_raa(ax[i, j], blk, "oo_vac", edges, f"{EST[est]}: {lab}")
            ax[-1, j].set_xlabel(xl)
        ax[i, 0].set_ylabel("$R_{AA}^{vac}$ (selection)")
    ax[0, 0].legend(fontsize=7, ncol=2)
    fig.suptitle("Figure 1. Selection bias: R_AA of oo_vac per class against N_coll × pp_ref", fontsize=11)
    fig.tight_layout(); plt.show()
else:
    print("no R_AA (needs oo_vac and pp_ref)")

## 2. Quenching in the classes, and against the true geometry

Q^c = R_AA^c(`oo_med`) / R_AA^c(`oo_vac`): the effect of the medium on the jets of class c, with the
selection divided out. Q_b (impact-parameter classes) is the quenching at the true geometry; Q_mult
(EPD, MBD) what the measured classes show.

In [ ]:
def q_block(est, key):
    blk = R["raa"][est][key]
    if "oo_med" not in blk or "oo_vac" not in blk:
        return None
    a, ea = np.asarray(blk["oo_med"]["raa"], float), np.asarray(blk["oo_med"]["err"], float)
    b, eb = np.asarray(blk["oo_vac"]["raa"], float), np.asarray(blk["oo_vac"]["err"], float)
    with np.errstate(divide="ignore", invalid="ignore"):
        q = a / b
        return blk["classes"], q, np.abs(q) * np.sqrt((ea / a) ** 2 + (eb / b) ** 2)

if R["raa"]:
    fig, ax = plt.subplots(len(EST), len(OBS), figsize=(16, 3.2 * len(EST)), sharey=True)
    for i, est in enumerate(EST):
        for j, (key, (lab, xl, edges)) in enumerate(OBS.items()):
            qb = q_block(est, key)
            if qb is None:
                continue
            cl, q, e = qb; x, xe = xc(edges)
            for c, (lo, hi) in enumerate(cl):
                ax[i, j].errorbar(x + (c - 2.5) * 0.08 * xe, q[c], e[c], xe, fmt="o", ms=3, color=COLS[c], label=f"{lo}-{hi}%")
            ax[i, j].axhline(1, color="k", lw=0.8); ax[i, j].set_ylim(0.3, 1.5); ax[i, j].set_title(f"{EST[est]}: {lab}", fontsize=9)
            ax[-1, j].set_xlabel(xl)
        ax[i, 0].set_ylabel("Q = R_AA(med) / R_AA(vac)")
    ax[0, 0].legend(fontsize=7, ncol=2)
    fig.suptitle("Figure 2. Quenching per class (selection divided out); last row: the true geometry (Q_b)", fontsize=11)
    fig.tight_layout(); plt.show()

## 3. Total: R_AA and R_CP of the quenched jets (the quantities the experiments measure)

R_AA^c(`oo_med`) = selection × quenching. R_CP between a central and a peripheral class cancels the
p+p reference; sPHENIX quotes 0–20 / 60–80 % (MBD), STAR 0–10 / 40–60 % (EPD).

In [ ]:
PAIRS = {"mbd": [((0, 20), (60, 80)), ((0, 20), (40, 60))], "epd": [((0, 10), (40, 60)), ((0, 10), (60, 80))]}
def rcp(est, key, fam, a, b):
    blk = R["raa"][est][key]
    if fam not in blk:
        return None
    cl = [tuple(c) for c in blk["classes"]]
    ia, ib = cl.index(a), cl.index(b)
    y, e = np.asarray(blk[fam]["raa"], float), np.asarray(blk[fam]["err"], float)
    with np.errstate(divide="ignore", invalid="ignore"):
        r = y[ia] / y[ib]
        return r, np.abs(r) * np.sqrt((e[ia] / y[ia]) ** 2 + (e[ib] / y[ib]) ** 2)

if R["raa"]:
    fig, ax = plt.subplots(2, len(OBS), figsize=(16, 6.5), sharey=True)
    for i, est in enumerate(("mbd", "epd")):
        for j, (key, (lab, xl, edges)) in enumerate(OBS.items()):
            x, xe = xc(edges)
            for k, (a, b) in enumerate(PAIRS[est]):
                for fam, mk, col in (("oo_vac", "s", "C0"), ("oo_med", "o", "C3")):
                    r = rcp(est, key, fam, a, b)
                    if r is not None:
                        ax[i, j].errorbar(x + (k - 0.5) * 0.15 * xe, r[0], r[1], xe, fmt=mk, ms=3, color=col,
                                          alpha=1 if k == 0 else 0.45,
                                          label=f"{fam} {a[0]}-{a[1]}/{b[0]}-{b[1]}%")
            ax[i, j].axhline(1, color="k", lw=0.8); ax[i, j].set_ylim(0.3, 1.5); ax[i, j].set_title(f"{EST[est]}: {lab}", fontsize=9)
            ax[-1, j].set_xlabel(xl)
        ax[i, 0].set_ylabel("$R_{CP}$"); ax[i, 0].legend(fontsize=7)
    fig.suptitle("Figure 3. R_CP: vacuum (selection only, squares) and medium (selection × quenching, circles)", fontsize=11)
    fig.tight_layout(); plt.show()

### Against sPHENIX (preliminary)

sPHENIX's R = 0.4 jet R_CP and R_AA (sPH-CONF-JET-2026-02, MBD classes, 19–60 GeV) read from the
note's vector figures (`refs/sphenix_oo_jet`; preliminary; the lowest bins' statistical bars are
upper bounds). Model: the full jets with the holes + partners subtracted, vacuum (selection) and
medium (total).

In [ ]:
SPD = (R.get("data") or {}).get("sphenix")
if SPD and R["raa"]:
    pairs = (("rcp_vs_60_80", "red", (0, 20), (60, 80)), ("rcp_vs_60_80", "blue", (20, 40), (60, 80)),
             ("rcp_vs_60_80", "green", (40, 60), (60, 80)), ("rcp_vs_40_60", "red", (0, 20), (40, 60)),
             ("rcp_vs_40_60", "blue", (20, 40), (40, 60)))
    fig, ax = plt.subplots(1, 6, figsize=(20, 3.6), sharey=True)
    x, xe = xc(E["sph_jet"])
    for j, (key, col, a, b) in enumerate(pairs):
        rows = SPD[key][col]
        xd = np.array([(r_["lo"] + r_["hi"]) / 2 for r_ in rows]); wd = np.array([r_["hi"] - r_["lo"] for r_ in rows])
        v = np.array([r_["value"] for r_ in rows])
        ax[j].errorbar(xd, v, [[r_["stat_lo"] for r_ in rows], [r_["stat_hi"] for r_ in rows]], wd / 2, fmt="*", ms=8, color="k", label="sPHENIX prel.")
        for xx, w_, r_ in zip(xd, wd, rows):
            ax[j].add_patch(plt.Rectangle((xx - w_ / 2, r_["value"] - r_["sys_lo"]), w_, r_["sys_lo"] + r_["sys_hi"], color="k", alpha=0.15))
        for fam, mk, c_ in (("oo_vac", "s", "C0"), ("oo_med", "o", "C3")):
            r = rcp("mbd", "sph_jet", fam, a, b)
            if r is not None:
                ax[j].errorbar(x, r[0], r[1], xe, fmt=mk, ms=4, color=c_, label=fam)
        ax[j].axhline(1, color="k", lw=0.8); ax[j].set_ylim(0, 1.6); ax[j].set_xlabel("$p_{T,jet}$ [GeV]")
        ax[j].set_title(f"R_CP {a[0]}-{a[1]} / {b[0]}-{b[1]} % (MBD)", fontsize=9)
    rows = SPD["raa_0_100"]["black"]
    xd = np.array([(r_["lo"] + r_["hi"]) / 2 for r_ in rows]); wd = np.array([r_["hi"] - r_["lo"] for r_ in rows])
    ax[5].errorbar(xd, [r_["value"] for r_ in rows], [[r_["stat_lo"] for r_ in rows], [r_["stat_hi"] for r_ in rows]], wd / 2,
                   fmt="*", ms=8, color="k", label="sPHENIX prel.")
    for fam, mk, c_ in (("oo_vac", "s", "C0"), ("oo_med", "o", "C3")):
        blk = R["raa"]["mbd"]["sph_jet"]
        if fam in blk:
            ax[5].errorbar(x, blk[fam]["raa_mb"], blk[fam]["err_mb"], xe, fmt=mk, ms=4, color=c_, label=fam)
    ax[5].axhline(1, color="k", lw=0.8); ax[5].set_title("R_AA 0-100 %", fontsize=9); ax[5].set_xlabel("$p_{T,jet}$ [GeV]")
    ax[0].set_ylabel("$R_{CP}$ / $R_{AA}$"); ax[0].legend(fontsize=8)
    fig.suptitle("Figure 3b. sPHENIX jets: selection (oo_vac) and total (oo_med) against the preliminary data", fontsize=11)
    fig.tight_layout(); plt.show()

### The medium's holes in the sPHENIX jets

The full jets subtract, at parton level, the four-momenta of the medium's holes (`partons/`, pstat −1)
and of their colour partners' copies (`hadronization_input/`, the hole's three-momentum) inside
ΔR < 0.4: iColored hadronizes recoil + partner, the correct balance is recoil − hole. No negative
hadronization is needed for a jet's pT (it is for hadron-level observables). Below: the quenched R_CP
with and without the subtraction (the vacuum arm has no holes).

In [ ]:
if R["raa"]:
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
    x, xe = xc(E["sph_jet"])
    for i, (est, (a, b)) in enumerate((("mbd", ((0, 20), (60, 80))), ("epd", ((0, 10), (40, 60))))):
        for key, mk, col, lab in (("sph_jet", "o", "C3", "holes + partners subtracted"), ("sph_jet_raw", "^", "0.5", "raw")):
            r = rcp(est, key, "oo_med", a, b)
            if r is not None:
                ax[i].errorbar(x, r[0], r[1], xe, fmt=mk, ms=4, color=col, label=lab)
        ax[i].axhline(1, color="k", lw=0.8); ax[i].set_ylim(0.3, 1.5); ax[i].set_xlabel("$p_{T,jet}$ [GeV]")
        ax[i].set_title(f"{EST[est]}: oo_med R_CP {a[0]}-{a[1]} / {b[0]}-{b[1]} %", fontsize=9)
    ax[0].set_ylabel("$R_{CP}$"); ax[0].legend(fontsize=8)
    fig.tight_layout(); plt.show()

## 4. STAR's I_CP: h–h and h+jet

STAR's definitions (arXiv:2604.13935): one charged trigger 7–30 GeV (|η| < 1.5, chosen at random if
several); h–h associated 3–7 GeV, near |Δφ| < 0.8, recoil |Δφ − π| < 1, flat background from
0.8 < |Δφ| < 1.6 (STAR's v_n = 0 variation); h+jet recoil charged jets R = 0.2, 0.5, |Δφ − π| < π/4,
8–19 GeV. In the model the hard event alone is used (no ρA, no mixed events). The recoil charged
jets subtract the medium's holes and partner copies in the cone with the charged fraction 2/3 of
their transverse momentum (grey triangles: without); the trigger and the h–h associated hadrons are
hadron-level and not subtracted: the h–h I_CP needs `negative_hadrons.py` (see the note above). Per-trigger yields
are N_coll-free; the selection still enters through the trigger's class assignment. Points: the
model's vacuum (selection) and medium (total) arms; stars: STAR's 0–10 % values (text of the paper;
the full HEPData tables to be added).

In [ ]:
ic = R.get("icp") or {}
STD = (R.get("data") or {}).get("star", {})
def star_cols(tab):
    # {column name: (x labels, values array [value, stat, syst])}
    return {c["name"]: (tab["x"], np.asarray(c["values"], float)) for c in tab["cols"]}

if ic and ("oo_vac" in ic or "oo_med" in ic):
    cl = ic["classes"]; labs = [f"{a}-{b}%" for a, b in cl]
    print("windows used for the triggers:", ic.get("windows_used"))
    fig, ax = plt.subplots(1, 4, figsize=(16, 3.8), sharey=True)
    data_cols = {}
    if STD:
        hh, hj = star_cols(STD["icp_hh_vs_ea"]), star_cols(STD["icp_hjet_vs_ea"])
        data_cols = {"near": [v for k, v in hh.items() if "near" in k][0], "recoil": [v for k, v in hh.items() if "recoil" in k][0],
                     "hjet_R0.2_8_19": [v for k, v in hj.items() if "0.2" in k][0], "hjet_R0.5_8_19": [v for k, v in hj.items() if "0.5" in k][0]}
    for j, obs in enumerate(("near", "recoil", "hjet_R0.2_8_19", "hjet_R0.5_8_19")):
        for k, (fam, mk, col) in enumerate((("oo_vac", "s", "C0"), ("oo_med", "o", "C3"))):
            if fam in ic:
                v = np.asarray(ic[fam]["icp"][obs], float)
                ax[j].errorbar(np.arange(len(cl)) + (k - 0.5) * 0.15, v[:, 0], v[:, 1], fmt=mk, color=col, label=fam)
                raw = obs.replace("_8_19", "_raw_8_19")
                if fam == "oo_med" and raw != obs and raw in ic[fam]["icp"]:
                    v = np.asarray(ic[fam]["icp"][raw], float)
                    ax[j].errorbar(np.arange(len(cl)) + 0.2, v[:, 0], v[:, 1], fmt="^", color="0.6", ms=4,
                                   label="oo_med, holes not subtracted")
        if obs in data_cols:
            xl, vals = data_cols[obs]
            pos = [labs.index(x.replace("–", "-") + "%") for x in xl]
            ax[j].errorbar(np.asarray(pos) - 0.25, vals[:, 0], vals[:, 1], fmt="*", ms=10, color="k", label="STAR (stat)")
            for p_, v_ in zip(pos, vals):
                ax[j].add_patch(plt.Rectangle((p_ - 0.33, v_[0] - v_[2]), 0.16, 2 * v_[2], color="k", alpha=0.2))
        ax[j].set_xticks(range(len(cl))); ax[j].set_xticklabels(labs, rotation=45, fontsize=8)
        ax[j].axhline(1, color="k", lw=0.8); ax[j].set_ylim(0.4, 1.6); ax[j].set_title(obs, fontsize=9)
    ax[0].set_ylabel("$I_{CP}$ (class / 40-60 %)"); ax[1].legend(fontsize=8)
    fig.suptitle("Figure 4. I_CP in STAR's EPD classes: vacuum arm = selection, medium arm = total; STAR (HEPData; boxes: syst)", fontsize=11)
    fig.tight_layout(); plt.show()
    for fam in ("oo_vac", "oo_med"):
        if fam in ic:
            print(fam, "I_AA (0-100 % over p+p):", {k: tuple(round(x, 3) for x in v) for k, v in ic[fam]["iaa"].items()})
else:
    print("no I_CP yet")

**Recoil jets vs jet pT.** STAR's Fig. 6 (I_CP 0–10 / 40–60 % per jet-pT bin) and Fig. 5 (the
per-trigger recoil-jet spectra per class, $(1/N_{trig})\,d^2N/dp_T d\eta$, with
Δη = 2 (1.5 − R)). The model's spectra are the hard event's charged jets with 2/3 of the holes +
partners subtracted, no unfolding needed (no detector).

In [ ]:
he = E["hjet"]; x, xe = xc(he)
if ic and STD:
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
    pt = STD["icp_hjet_vs_pt"]
    for j, R_ in enumerate((0.2, 0.5)):
        c = [cc for cc in pt["cols"] if cc["qual"].get("RJET") == str(R_)][0]
        v = np.asarray(c["values"], float); lo_hi = np.asarray(pt["x"], float)
        xd = lo_hi.mean(1)
        ax[j].errorbar(xd, v[:, 0], v[:, 1], 0.5 * np.diff(lo_hi, axis=1).ravel(), fmt="*", ms=9, color="k", label="STAR")
        for xx, w_, vv in zip(xd, np.diff(lo_hi, axis=1).ravel(), v):
            ax[j].add_patch(plt.Rectangle((xx - w_ / 2, vv[0] - vv[2]), w_, 2 * vv[2], color="k", alpha=0.15))
        for k, (fam, mk, col) in enumerate((("oo_vac", "s", "C0"), ("oo_med", "o", "C3"))):
            if fam in ic:
                m = np.asarray(ic[fam][f"icp_vs_pt_hjet_R{R_}"], float)
                ax[j].errorbar(x + (k - 0.5) * 0.2, m[:, 0], m[:, 1], xe, fmt=mk, color=col, label=fam)
        ax[j].axhline(1, color="k", lw=0.8); ax[j].set_ylim(0.2, 1.6); ax[j].set_xlabel("$p_{T,jet}^{ch}$ [GeV]")
        ax[j].set_title(f"recoil-jet I_CP 0-10 / 40-60 %, R = {R_}", fontsize=9)
    ax[0].set_ylabel("$I_{CP}$"); ax[0].legend(fontsize=8)
    fig.tight_layout(); plt.show()

    fig, ax = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
    ccl = [tuple(c) for c in ic["classes"]]
    for j, R_ in enumerate((0.2, 0.5)):
        tab = STD[f"hjet_spectra_R{R_}"]; lo_hi = np.asarray(tab["x"], float); xd = lo_hi.mean(1)
        deta = 2 * (1.5 - R_)
        for k, c in enumerate(tab["cols"]):
            ea = c["qual"].get("EA interval").replace("–", "-").replace("%", ""); v = np.asarray(c["values"], float)
            ax[j].errorbar(xd, v[:, 0] * 10 ** (4 - k), v[:, 1] * 10 ** (4 - k), fmt="*", ms=7, color=COLS[k], label=f"STAR {ea}% x10^{4-k}")
            a, b = (int(t) for t in ea.split("-"))
            if "oo_med" in ic and (a, b) in ccl:
                y = np.asarray(ic["oo_med"]["yields"][ccl.index((a, b))][f"hjet_R{R_}"], float)
                ax[j].errorbar(x, y[:, 0] / (2 * xe) / deta * 10 ** (4 - k), y[:, 1] / (2 * xe) / deta * 10 ** (4 - k), xe,
                               fmt="o", mfc="none", ms=5, color=COLS[k])
        ax[j].set_yscale("log"); ax[j].set_xlabel("$p_{T,jet}^{ch}$ [GeV]"); ax[j].set_title(f"R = {R_}: STAR (stars) and oo_med (open circles)", fontsize=9)
    ax[0].set_ylabel("$(1/N_{trig})\\, d^2N/dp_T d\\eta$  [GeV$^{-1}$]"); ax[1].legend(fontsize=7)
    fig.tight_layout(); plt.show()

## 5. The p+p reference with the jet in its own medium (3c)

`pp_med` / `pp_ref`: the model's quenching in p+p together with Matter's lower vacuum cutoff (QS 0.9
GeV in the medium arm against Q0 2 GeV in the vacuum arm). Expected close to 1.

In [ ]:
pr = R.get("pp_med_over_ref")
if pr:
    fig, ax = plt.subplots(1, len(OBS), figsize=(16, 3.2), sharey=True)
    for j, (key, (lab, xl, edges)) in enumerate(OBS.items()):
        x, xe = xc(edges)
        ax[j].errorbar(x, pr[key]["ratio"], pr[key]["err"], xe, fmt="o", ms=3)
        ax[j].axhline(1, color="k", lw=0.8); ax[j].set_ylim(0.6, 1.4); ax[j].set_title(lab, fontsize=9); ax[j].set_xlabel(xl)
    ax[0].set_ylabel("pp_med / pp_ref")
    fig.tight_layout(); plt.show()
else:
    print("pp_med not run yet (3c)")

## 6. Systematics

- **Holes in hadron-level observables**: charged-hadron R_AA / R_CP and the h–h I_CP of the
  medium arm until `negative_hadrons.py` is run on the phase-3 medium files.
- **The 2/3 charged fraction** for the charged jets' hole subtraction (an approximation; the
  `*_raw` spectra bound the effect).
- **Matter's vacuum cutoff (QS)**: `pp_med`'s events without a medium against `pp_ref`'s, per window
  (the table below); quoted as the systematic on every medium/vacuum ratio.
- **MPI**: the frozen phase-2 arms with the old MPI and with MPI off (`g2_mpi_check.ipynb`); for
  the forward classes ±0.05 (MBD) / ±0.06 (EPD) on the vacuum R_CP.
- **Estimator**: EPD vs MBD vs mid-rapidity (Figures 1–3).
- **The flat UE** (`ue_trend_cv.ipynb`): mid-rapidity estimators carry the UE trend (−13 % at 11–20
  GeV, +6–19 % above, against STAR).
- **The energy-conservation trend** of the forward activity (`fwd_check.py`): the forward selection
  share is an upper estimate until constrained by STAR's p+Au event activity.

In [ ]:
qs = R.get("qs_check")
if qs:
    for key, rows in qs.items():
        print(key)
        for r in rows:
            print(f"  window {r['window'][0]:g}-{r['window'][1]:g}: no-medium events {r['n_med']} / {r['n_ref']}, "
                  f"total per event pp_med/pp_ref {r['total']:.3f} +- {r['total_err']:.3f}")
else:
    print("needs pp_med (3c)")

## 7. Reading

To be written when phase 3 (and 3b, 3c) are complete: the selection share and the quenching share
per observable and estimator, with the systematics above, against STAR's I_CP and sPHENIX's R_CP.
Model results, not vetted physics results.